# Stage C jobs on Colab

Runtime: T4 GPU. Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` must exist in this
account's Colab Secrets with notebook access on. Outputs land on Drive under
`MyDrive/medseg-label-efficiency/work/outputs/<arm>/`, the same layout as the
laptop's `outputs/`. Keep the tab open; a lost session resumes from Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os

from google.colab import userdata

# secrets are readable only in this kernel process; a %%bash cell and the
# scripts it starts inherit the environment, so hand them over here
for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    os.environ[name] = userdata.get(name)


In [ ]:
%%bash
set -e
if [ ! -d /content/repo ]; then
  git clone --depth 1 https://github.com/ahmadM9/medseg-label-efficiency.git /content/repo
fi
cd /content/repo && git pull --ff-only
pip install -q -e /content/repo kaggle
python /content/repo/colab/prepare_colab.py

In [ ]:
%%bash
# edit --only for the session: zeroshot,large_ft,reeval_unet,reeval_ft,reeval_head,incontext
# add --limit 4 for a smoke run
export LAUNCH_REPO=/content/repo
export LAUNCH_INPUT=/content/input
export LAUNCH_WORK=/content/drive/MyDrive/medseg-label-efficiency/work
export LAUNCH_MLFLOW=sqlite:////content/mlflow.db
cd /content/repo
python kaggle/scale/run_scale.py --only zeroshot --limit 4
cp -f /content/mlflow.db "$LAUNCH_WORK/mlflow.db" 2>/dev/null || true
ls "$LAUNCH_WORK/outputs"